# endgame-probe -- how does eat-rest-v1 die?

`tune_v1` found v1's numeric settings flat (80 variants, none better on fresh seeds), so the next gain has to come from a
behaviour change. This runs `external/candidates/endgame_probe.py`: the untouched baseline on fresh seeds (11000+), recording
a 50 s timeline of the world and the colony plus every death and birth, and prints what changes in the run-up to extinction.

16 games, roughly 5-8 minutes; runs in the foreground so the report lands in the cell. Resumable: rerunning skips finished
games, and raising `SEEDS` only plays the new ones. Mechanism study: nothing is written to `results/`.

**Cluster setup:** same as `parameter-tuning.ipynb` (`.env` with `GITHUB_TOKEN=<token>`).

In [7]:
import os

CLONE_DIR = "/home/jovyan/Nordic-AI-cup-2026"
if os.path.isdir(os.path.join(CLONE_DIR, ".git")):
    print(f"{CLONE_DIR} already cloned - skipping (use `git pull` there to update)")
else:
    # GitHub token is read from a git-ignored .env (GITHUB_TOKEN=...) in the kernel's cwd, or from the environment
    if os.path.isfile(".env"):
        for line in open(".env"):
            key, sep, value = line.strip().partition("=")
            if sep and not key.startswith("#"):
                os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))
    TOKEN = os.environ.get("GITHUB_TOKEN")
    if not TOKEN:
        raise RuntimeError(f"GITHUB_TOKEN not set - create {os.path.abspath('.env')} containing GITHUB_TOKEN=<token>")
    !git clone https://{TOKEN}@github.com/sjoeen/Nordic-AI-cup-2026.git {CLONE_DIR}

/home/jovyan/Nordic-AI-cup-2026 already cloned - skipping (use `git pull` there to update)


In [8]:
import glob
import os
import subprocess
import sys

os.chdir(CLONE_DIR)
!git fetch origin challenge-1V2
!git checkout challenge-1V2
!git pull origin challenge-1V2

# Must run from survival-simulator/ so `src`, `agents`, `training` import.
if os.path.basename(os.getcwd()) != "survival-simulator":
    candidates = sorted({os.path.realpath(p) for p in glob.glob(os.path.join(os.getcwd(), "**", "survival-simulator"), recursive=True)
                         if os.path.isfile(os.path.join(p, "requirements.txt"))})
    if len(candidates) != 1:
        raise RuntimeError(f"cwd is {os.getcwd()}; found {len(candidates)} survival-simulator checkouts {candidates} - %cd into the right one")
    os.chdir(candidates[0])

print("cwd:", os.getcwd())
sys.path.insert(0, os.getcwd())
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

remote: Enumerating objects: 19, done.
remote: Counting objects: 100% (19/19), done.
remote: Compressing objects: 100% (3/3), done.
remote: Total 11 (delta 7), reused 11 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (11/11), 7.60 KiB | 707.00 KiB/s, done.
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
   a1e6b54..32f4c04  challenge-1V2 -> origin/challenge-1V2
M	Nordic-AI-Cup-2026-main/survival-simulator/results/index.csv
Already on 'challenge-1V2'
Your branch is behind 'origin/challenge-1V2' by 1 commit, and can be fast-forwarded.
  (use "git pull" to update your local branch)
From https://github.com/sjoeen/Nordic-AI-cup-2026
 * branch            challenge-1V2 -> FETCH_HEAD
Updating a1e6b54..32f4c04
Fast-forward
 .../survival-simulator/endgame-probe.ipynb         | 179 ++++++++++++++++++---
 .../candidates/eat-rest-endgame/survival_agent.py  | 150 +++++++++++++++++
 .../external/candidates/endgame_probe.py           |  67 

In [9]:
!{sys.executable} -m pip install -r requirements.txt -r requirements-dev.txt

## Run the probe and print the report

In [10]:
SEEDS = 16
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds {SEEDS} 2>&1 | grep -v "pkg_resources\|pygame"

=== endgame probe: original-eat-rest-preserved, 16 games to run, 0 stored, overrides {} ===
  [  2.2 min] 8/16 games
  [  4.0 min] 16/16 games

16 games   extinction: mean 1197  sd 223  min 830  median 1113  max 1594   score mean 1185

-- world and colony by clock time (mean over the games still alive then)
     t games            n         pred        awake        ratio       sensed   threatened      endgame        trees       fruits fruit_energy       energy          age        speed
     0    16         5.00         0.00         0.00         0.00         0.00         0.00         0.00        25.94        32.06       647.25       150.15         0.10        10.00
   250    16         9.06         1.44         1.25         0.16         0.07         0.07         0.00        60.25       125.94      6400.50       216.27        67.13        10.91
   500    16         7.25         4.44         3.25         0.64         0.22         0.21         0.00        44.62        81.12      4074.31   

## Endgame mode trial: `eat-rest-endgame` against the baseline on the same seeds

`external/candidates/eat-rest-endgame` is v1 plus an endgame mode (see its docstring). It reads the colony's own state:
ON when the smoothed mean energy drops under 175 or 3 or fewer agents are left (never before t=400), OFF again once the
colony has held mean energy above 205 with 5+ agents for 30 s. While ON it vetoes births that would leave the parent under
50 energy or happen with a predator within 130, and agents of 95 s or older are ordered to breed before old age takes their
energy. Run the baseline cell above first; `--compare` then prints the paired per-seed difference, when the mode first
switched on (`eg_first_on`), how often it switched on/off, and how often each rule fired.

v2 rules (after the first trial, +49 SE 63 on 16 seeds): an old agent is only ordered to breed with a fruit within 120 and no
predator within 130. The games land in `logs/eg2/eg_v2`; the v1-rule games from the first trial stay in `logs/eg2/eg`.


In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/eg_v2 --candidate eat-rest-endgame --seeds {SEEDS} --compare logs/eg2/base 2>&1 | grep -v "pkg_resources\|pygame"

## Variants of the endgame mode (each in its own folder, same seeds, compared with the baseline)

`--set` overrides the candidate's settings. `eg_overrides` holds baseline settings that apply only while the endgame is on.

In [ ]:
VARIANTS = {   # the default run above is v2: ordered births by old agents only next to food and away from predators
    "legacy_v1":   '{"eg_legacy_food": 0}',                                      # v1 rule: ordered births anywhere (the +49 run)
    "nolegacy":    '{"eg_legacy_age": 0}',                                       # vetoes only, no ordered births at all
    "food":        '{"eg_birth_food": 120}',                                     # EVERY endgame birth needs a fruit within 120
    "stay":        '{"eg_overrides": {"dispersal_time": 0}}',                    # endgame newborns do not walk away from the birth spot
    "food_stay":   '{"eg_birth_food": 120, "eg_overrides": {"dispersal_time": 0}}',
}
for name, overrides in VARIANTS.items():
    !{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/{name} --candidate eat-rest-endgame --seeds {SEEDS} --set '{overrides}' --compare logs/eg2/base 2>&1 | grep -v "pkg_resources\|pygame" | grep -A12 "paired against"

## Report only (no new games)

In [ ]:
!{sys.executable} -u external/candidates/endgame_probe.py --out logs/eg2/base --seeds 0 2>&1 | grep -v "pkg_resources\|pygame"